# FAISS search settings: controlled ablation
Compare 64/128/256 searched partitions × 200/400/800 initial neighbors. Keep the same 500 tuning businesses, the final **20 candidates**, saved classifier, and standard-profile cutoff. No new embeddings, retraining, GPU, or test predictions.

Attach the original full_pool_embedding_v1 and completed ml_deadline_upgrade_v2 outputs. Run all cells in a separate notebook. Alternatively append the code cells to the completed diagnostic session; its local cache is reused if present. Do not run concurrently with another expensive notebook job.

This is a tuning experiment. Do not interpret the best configuration as an independently validated improvement. Later validation must use untouched businesses. Timing is measured after warm-up with randomized configuration order and a repeated baseline; it still reflects this machine and cache state.


In [ ]:
%pip install -q numpy==2.0.2 pandas==2.3.3 scikit-learn==1.6.1 faiss-cpu==1.11.0 threadpoolctl


In [ ]:
from pathlib import Path
import os, sys, json, pickle, shutil, time, hashlib, gc, zipfile
import numpy as np
import pandas as pd
from threadpoolctl import threadpool_limits

# Set explicit paths if more than one matching saved output is attached.
UPGRADE_PATH = ""
CACHE_PATH = ""
N_TUNE = 500
EXACT_QUERY_LIMIT = 10
ROOT = Path('/kaggle/working/faiss_search_ablation_v1')
OUT = ROOT / 'results'
OUT.mkdir(parents=True, exist_ok=True)

def discover(explicit, marker, required):
    paths = [Path(explicit)] if explicit else sorted({p.parent for p in Path('/kaggle/input').rglob(marker)})
    paths = [p for p in paths if all((p / n).exists() for n in required)]
    assert len(paths) == 1, f'Set explicit path; found {paths}'
    return paths[0]

UPGRADE = discover(UPGRADE_PATH, 'upgrade_report.json',
    ['sample.pkl', 'queries.npy', 'model/matcher.pkl', 'model/experiment_results.json'])
SOURCE = discover(CACHE_PATH, 'encoding_complete.json',
    ['vectors.f16', 'records.sqlite', 'index_catalog.json', 'sample.tsv', 'configuration.json'])
SAVED_CODE = UPGRADE.parent / 'code'
assert (SAVED_CODE/'submit_pipeline.py').exists()
sys.path.insert(0, str(SAVED_CODE))
import embedding_core as c
import submit_pipeline as s
bundle, model_meta = s.load_model(UPGRADE/'model')
report = json.loads((UPGRADE/'upgrade_report.json').read_text())
threshold = float(report['tuning_options']['standard']['threshold'])
with (UPGRADE/'sample.pkl').open('rb') as f:
    sample, truth = pickle.load(f)
queries = np.load(UPGRADE/'queries.npy', mmap_mode='r')
eligible = np.flatnonzero(sample['split'].to_numpy() == 'tune')
chosen = np.random.default_rng(20260928).choice(eligible, min(N_TUNE,len(eligible)), replace=False)
frame = sample.iloc[chosen].reset_index(drop=True)
q = np.ascontiguousarray(queries[chosen], dtype=np.float32)
assert np.allclose(np.linalg.norm(q,axis=1),1,atol=1e-4)
assert len(frame) and frame.entity_id.is_unique
frame.to_csv(OUT/'audit_businesses.tsv',sep='\t',index=False)
print('Saved upgrade:',UPGRADE)
print('Training cache:',SOURCE)
print('Tuning businesses:',len(frame),'fixed cutoff:',threshold)


In [ ]:
# Copy immutable cache files locally; never alter the attached source.
previous_cache = Path('/kaggle/working/entity_gap_audit_v1/local_cache')
CACHE = previous_cache if (previous_cache/'copy_identity.json').exists() else ROOT/'local_cache'
CACHE.mkdir(exist_ok=True)
catalog = json.loads((SOURCE/'index_catalog.json').read_text())
names = ['vectors.f16','records.sqlite','encoding_complete.json','index_catalog.json'] + [x['file'] for x in catalog]
manifest = {'source':str(SOURCE), 'sizes':{n:(SOURCE/n).stat().st_size for n in names}}
identity = CACHE/'copy_identity.json'
if identity.exists():
    assert json.loads(identity.read_text()) == manifest, 'Different source cache; use a new ROOT.'
else:
    assert not any(CACHE.iterdir()), 'Unknown files in cache directory; use a new ROOT.'
    identity.write_text(json.dumps(manifest))
need = sum(manifest['sizes'][n] for n in names if not (CACHE/n).exists())
assert shutil.disk_usage(CACHE).free > need + 2*2**30, 'Need enough disk for cache plus 2 GiB.'
for name in names:
    dest=CACHE/name
    if dest.exists():
        assert dest.stat().st_size == manifest['sizes'][name]
        continue
    print('Copying',name,flush=True)
    temp=dest.with_name(dest.name+'.partial')
    shutil.copyfile(SOURCE/name,temp)
    assert temp.stat().st_size == manifest['sizes'][name]
    os.replace(temp,dest)
print('Local cache ready.')


In [ ]:
# Truth is used only for scoring/diagnosis, never to choose search candidates.
searcher = s.TestSearcher(CACHE,64,200)
searcher.faiss.omp_set_num_threads(4)
wanted = set().union(*(truth[sid] for sid in frame.entity_id))
truth_records = {}
country_codes = {x['country']:i for i,x in enumerate(catalog)}
row_country = np.full(searcher.total,-1,dtype=np.int16)
started=time.perf_counter()
cursor=searcher.db.execute('SELECT rid,entity_id,business_name,business_address,country FROM records')
while True:
    batch=cursor.fetchmany(100000)
    if not batch: break
    for rid,eid,name,address,country in batch:
        row_country[rid]=country_codes[c.norm(country)]
        if eid in wanted: truth_records[eid]=(rid,(eid,name,address,country))
assert set(truth_records)==wanted, 'Missing truth IDs in the attached training cache.'
assert (row_country>=0).all()
print('Truth lookup complete:',len(wanted),'links;',round(time.perf_counter()-started,1),'seconds')


In [ ]:
def retrieve(nprobe, overfetch):
    ids=np.full((len(frame),overfetch),-1,np.int64)
    sims=np.full((len(frame),overfetch),-np.inf,np.float32)
    timing={'ann_seconds':0.,'vector_gather_rerank_seconds':0.}
    countries=frame.country.map(c.norm).to_numpy()
    for country in np.unique(countries):
        index=searcher.indexes[country]
        index.nprobe=min(nprobe,index.nlist)
        owners=np.flatnonzero(countries==country)
        for start in range(0,len(owners),32):
            ix=owners[start:start+32]
            t=time.perf_counter()
            _,raw=index.search(np.ascontiguousarray(q[ix]),overfetch)
            timing['ann_seconds']+=time.perf_counter()-t
            t=time.perf_counter()
            valid=raw>=0
            vectors=np.asarray(searcher.vectors[np.maximum(raw,0)],dtype=np.float32)
            vectors/=np.maximum(np.linalg.norm(vectors,axis=2,keepdims=True),1e-12)
            cosine=np.einsum('bkd,bd->bk',vectors,q[ix],optimize=False)
            for j,i in enumerate(ix):
                positions=np.flatnonzero(valid[j])
                order=positions[np.lexsort((raw[j,positions],-cosine[j,positions]))]
                ids[i,:len(order)]=raw[j,order]
                sims[i,:len(order)]=cosine[j,order]
            timing['vector_gather_rerank_seconds']+=time.perf_counter()-t
    return ids,sims,timing

def evaluate(ids, probabilities, labels, k):
    valid=ids[:,:k]>=0
    accepted=valid & (probabilities[:,:k]>=threshold)
    hits=(valid & labels[:,:k]).sum(axis=1)
    tp=(accepted & labels[:,:k]).sum(axis=1)
    fp=(accepted & ~labels[:,:k]).sum(axis=1)
    counts=np.array([len(truth[x]) for x in frame.entity_id])
    fn=counts-tp
    denom=1.25*tp+fp+.25*fn
    score=np.divide(1.25*tp,denom,out=np.zeros(len(frame)),where=denom!=0)
    score[(counts==0)&(accepted.sum(axis=1)==0)]=1
    return dict(k=k,true_links=int(counts.sum()),retrieved_true=int(hits.sum()),
        recall=float(hits.sum()/max(counts.sum(),1)),macro_F05=float(score.mean()),
        true_accepted=int(tp.sum()),false_accepted=int(fp.sum()),
        missed_search=int((counts-hits).sum()),retrieved_rejected=int((hits-tp).sum()),
        mean_candidates=float(valid.sum(axis=1).mean()),
        singletons=int((counts==0).sum()),singletons_correct=int(((counts==0)&(accepted.sum(axis=1)==0)).sum()))



In [ ]:
# Freeze settings and provenance for this experiment.
import sklearn
config={'source':str(SOURCE),'upgrade':str(UPGRADE),'query_indices':chosen.tolist(),
        'threshold':threshold,'final_k':20,'probes':[64,128,256],'overfetch':[200,400,800],
        'model_sha256':hashlib.sha256((UPGRADE/'model/matcher.pkl').read_bytes()).hexdigest(),
        'runner_sha256':hashlib.sha256((SAVED_CODE/'submit_pipeline.py').read_bytes()).hexdigest(),
        'core_sha256':hashlib.sha256((SAVED_CODE/'embedding_core.py').read_bytes()).hexdigest(),
        'sklearn_version':sklearn.__version__}
config_file=OUT/'configuration.json'
if config_file.exists():
    assert json.loads(config_file.read_text())==config, 'Configuration changed; use a new ROOT.'
else: config_file.write_text(json.dumps(config,indent=2))

def score_top20(ids,sims):
    t=time.perf_counter()
    records=searcher.fetch(ids[:,:20])
    lookup=time.perf_counter()-t
    t=time.perf_counter()
    probabilities=np.full((len(frame),20),np.nan)
    labels=np.zeros((len(frame),20),bool)
    xs=[]; slots=[]
    for i,left in enumerate(frame[c.COLS].itertuples(index=False,name=None)):
        for j,rid in enumerate(ids[i,:20]):
            if rid<0: continue
            right=records[int(rid)]
            xs.append(c.features(left,right)+[float(sims[i,j])]);slots.append((i,j))
            labels[i,j]=right[0] in truth[left[0]]
    if xs:
        values=bundle['model'].predict_proba(np.asarray(xs,np.float32))[:,1]
        for (i,j),p in zip(slots,values): probabilities[i,j]=p
    return probabilities,labels,lookup,time.perf_counter()-t

settings=[(p,n) for p in (64,128,256) for n in (200,400,800)]
order=np.random.default_rng(83).permutation(len(settings))
runs=[('baseline_before',64,200)]+[(f'p{settings[i][0]}_n{settings[i][1]}',*settings[i]) for i in order]+[('baseline_after',64,200)]
rows=[]; slice_rows=[]
with threadpool_limits(limits=4):
    # Warm each profile once. This is not counted as production throughput.
    print('Warming search profiles (9 short sample searches)...',flush=True)
    saved_frame,saved_q=frame,q
    try:
        frame=frame.iloc[:32].copy();q=q[:32]
        for p,n in settings: retrieve(p,n)
    finally:
        frame,q=saved_frame,saved_q
    for label,p,n in runs:
        checkpoint=OUT/f'{label}.npz'
        if checkpoint.exists():
            with np.load(checkpoint) as z:
                ids=z['ids'];sims=z['sims'];prob=z['prob'];labels=z['labels']
            timing=json.loads((OUT/f'{label}_timing.json').read_text())
            print('Reusing completed',label,flush=True)
        else:
            print('Searching',label,flush=True)
            c.prepared.cache_clear()
            ids,sims,timing=retrieve(p,n)
            prob,labels,lookup,features_seconds=score_top20(ids,sims)
            timing['record_lookup_seconds']=lookup
            timing['features_model_seconds']=features_seconds
            timing['total_seconds']=sum(timing.values())
            with checkpoint.with_suffix('.partial').open('wb') as f:
                np.savez_compressed(f,ids=ids,sims=sims,prob=prob,labels=labels)
            (OUT/f'{label}_timing.json').write_text(json.dumps(timing))
            os.replace(checkpoint.with_suffix('.partial'),checkpoint)
        metrics=evaluate(ids,prob,labels,20)
        row=dict(run=label,nprobe=p,overfetch=n,**metrics,**timing)
        rows.append(row)
        for missing in [False,True]:
            actual=found=accepted=0
            for i,sid in enumerate(frame.entity_id):
                for eid in truth[sid]:
                    rid,right=truth_records[eid]
                    if (not bool(str(right[2]).strip()))!=missing: continue
                    actual+=1
                    pos=np.flatnonzero(ids[i,:20]==rid)
                    if len(pos):
                        found+=1;accepted+=int(prob[i,pos[0]]>=threshold)
            slice_rows.append(dict(run=label,address_missing=missing,true_links=actual,retrieved=found,
                                   accepted=accepted,recall=found/max(actual,1)))
        pd.DataFrame(rows).to_csv(OUT/'search_ablation.csv',index=False)
        pd.DataFrame(slice_rows).to_csv(OUT/'missing_address_slices.csv',index=False)
        print({k:row[k] for k in ['run','recall','macro_F05','false_accepted','total_seconds']},flush=True)


In [ ]:
table=pd.DataFrame(rows)
grid=table[table.run.str.startswith('p')].copy().sort_values(['nprobe','overfetch'])
baseline=grid[(grid.nprobe==64)&(grid.overfetch==200)].iloc[0]
grid['extra_true_retrieved']=grid.retrieved_true-baseline.retrieved_true
grid['macro_F05_change']=grid.macro_F05-baseline.macro_F05
grid['time_ratio']=grid.total_seconds/baseline.total_seconds
# Show the accuracy/runtime tradeoff without automatically selecting or deploying a model.
grid['pareto_candidate']=[not ((grid.macro_F05>=r.macro_F05)&(grid.total_seconds<=r.total_seconds)&
    ((grid.macro_F05>r.macro_F05)|(grid.total_seconds<r.total_seconds))).any() for r in grid.itertuples()]
grid.to_csv(OUT/'comparison.csv',index=False)
print('FIXED 20 CANDIDATES; FIXED CLASSIFIER AND CUTOFF')
display(grid[['nprobe','overfetch','recall','macro_F05','extra_true_retrieved','false_accepted',
              'total_seconds','time_ratio','pareto_candidate']])
print('CACHE/TIMING STABILITY:')
display(table[table.run.str.startswith('baseline')][['run','ann_seconds','vector_gather_rerank_seconds','total_seconds']])
print('Partitions varied at each fixed neighbor count:')
display(grid.pivot(index='nprobe',columns='overfetch',values='recall'))
print('Interpretation: higher recall at fixed overfetch supports searching more partitions; higher recall at fixed nprobe supports fetching more neighbors. Both can contribute.')
print('These sample timings exclude encoding, index building, and output writing; do not extrapolate them as a guaranteed submission ETA.')
searcher.close()
archive=ROOT/'faiss_search_ablation_results.zip'
with zipfile.ZipFile(archive,'w',zipfile.ZIP_DEFLATED) as z:
    for path in sorted(OUT.iterdir()): z.write(path,path.name)
print('Download:',archive)
